In [3]:
import pandas as pd
import numpy as np

PATH = "US_Accidents_March23.csv"   # change to your file path
NROWS = None                        # set e.g. 500_000 to test quickly

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

In [4]:
# %% 2. Load (memory-efficient)
# Low-cardinality text columns are loaded as 'category' to save RAM.
cat_cols = [
    "Source", "Severity", "State", "Country", "Timezone", "Airport_Code",
    "Wind_Direction", "Weather_Condition", "Sunrise_Sunset", "Civil_Twilight",
    "Nautical_Twilight", "Astronomical_Twilight",
]
dtype_map = {c: "category" for c in cat_cols}

df = pd.read_csv(PATH, dtype=dtype_map, nrows=NROWS)

# Datetimes (some rows have fractional seconds, so use format="mixed")
for c in ["Start_Time", "End_Time", "Weather_Timestamp"]:
    df[c] = pd.to_datetime(df[c], format="mixed", errors="coerce")

# Downcast numeric columns
for c in df.select_dtypes("float64").columns:
    df[c] = pd.to_numeric(df[c], downcast="float")


In [5]:
# %% 3. Overview: size, memory, time span
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"Records (rows)     : {df.shape[0]:,}")
print(f"Attributes (cols)  : {df.shape[1]}")
print(f"Memory usage       : {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
print(f"Start_Time range   : {df['Start_Time'].min()}  ->  {df['Start_Time'].max()}")
print(f"Duplicate rows     : {df.duplicated().sum():,}")
print(f"Duplicate IDs      : {df['ID'].duplicated().sum():,}")


DATASET OVERVIEW
Records (rows)     : 7,728,394
Attributes (cols)  : 46
Memory usage       : 4.27 GB
Start_Time range   : 2016-01-14 20:18:33  ->  2023-03-31 23:30:00
Duplicate rows     : 0
Duplicate IDs      : 0


In [6]:
# %% 4. Column groups by data type
bool_cols = df.select_dtypes("bool").columns.tolist()
num_cols = df.select_dtypes(include=["number"]).columns.tolist()
dt_cols = df.select_dtypes("datetime").columns.tolist()
cat_like = df.select_dtypes(include=["category", "object"]).columns.tolist()

print("\nNumerical   :", len(num_cols), num_cols)
print("\nBoolean     :", len(bool_cols), bool_cols)
print("\nDatetime    :", len(dt_cols), dt_cols)
print("\nCategorical/Text:", len(cat_like), cat_like)



Numerical   : 12 ['Start_Lat', 'Start_Lng', 'End_Lat', 'End_Lng', 'Distance(mi)', 'Temperature(F)', 'Wind_Chill(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Speed(mph)', 'Precipitation(in)']

Boolean     : 13 ['Amenity', 'Bump', 'Crossing', 'Give_Way', 'Junction', 'No_Exit', 'Railway', 'Roundabout', 'Station', 'Stop', 'Traffic_Calming', 'Traffic_Signal', 'Turning_Loop']

Datetime    : 3 ['Start_Time', 'End_Time', 'Weather_Timestamp']

Categorical/Text: 18 ['ID', 'Source', 'Severity', 'Description', 'Street', 'City', 'County', 'State', 'Zipcode', 'Country', 'Timezone', 'Airport_Code', 'Wind_Direction', 'Weather_Condition', 'Sunrise_Sunset', 'Civil_Twilight', 'Nautical_Twilight', 'Astronomical_Twilight']


In [7]:
# %% 5. Schema table: dtype, missing, unique, example
schema = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "missing": df.isna().sum(),
    "missing_%": df.isna().mean() * 100,
    "unique": df.nunique(dropna=True),
    "example": df.iloc[0],
})
print("\nSCHEMA")
print(schema)


SCHEMA
                                dtype  non_null  missing  missing_%   unique                                            example
ID                             object   7728394        0      0.000  7728394                                                A-1
Source                       category   7728394        0      0.000        3                                            Source2
Severity                     category   7728394        0      0.000        4                                                  3
Start_Time             datetime64[ns]   7728394        0      0.000  5801064                                2016-02-08 05:46:00
End_Time               datetime64[ns]   7728394        0      0.000  6463024                                2016-02-08 11:00:00
Start_Lat                     float32   7728394        0      0.000  1900527                                             39.865
Start_Lng                     float32   7728394        0      0.000  1750819                    

In [8]:
# %% 6. Missing values (only columns that have any)
missing = schema.loc[schema["missing"] > 0, ["missing", "missing_%"]]
print("\nMISSING VALUES")
print(missing.sort_values("missing_%", ascending=False))



MISSING VALUES
                       missing  missing_%
End_Lat                3402762     44.029
End_Lng                3402762     44.029
Precipitation(in)      2203586     28.513
Wind_Chill(F)          1999019     25.866
Wind_Speed(mph)         571233      7.391
Visibility(mi)          177098      2.292
Wind_Direction          175206      2.267
Humidity(%)             174144      2.253
Weather_Condition       173459      2.244
Temperature(F)          163853      2.120
Pressure(in)            140679      1.820
Weather_Timestamp       120228      1.556
Sunrise_Sunset           23246      0.301
Civil_Twilight           23246      0.301
Nautical_Twilight        23246      0.301
Astronomical_Twilight    23246      0.301
Airport_Code             22635      0.293
Street                   10869      0.141
Timezone                  7808      0.101
Zipcode                   1915      0.025
City                       253      0.003
Description                  5      0.000


In [9]:
# %% 7. Numerical summary (+ skewness, kurtosis)
num_summary = df[num_cols].describe(percentiles=[.01, .25, .5, .75, .99]).T
num_summary["skew"] = df[num_cols].skew()
num_summary["kurtosis"] = df[num_cols].kurt()
print("\nNUMERICAL SUMMARY")
print(num_summary)


NUMERICAL SUMMARY
                          count    mean    std      min       1%      25%     50%     75%     99%       max   skew   kurtosis
Start_Lat         7,728,394.000  36.201  5.076   24.555   25.733   33.400  35.824  40.085  47.479    49.002 -0.072     -0.532
Start_Lng         7,728,394.000 -94.702 17.392 -124.624 -122.959 -117.219 -87.767 -80.354 -71.482   -67.113 -0.483     -1.363
End_Lat           4,325,632.000  36.262  5.273   24.566   25.687   33.462  36.183  40.179  47.274    49.075 -0.158     -0.557
End_Lng           4,325,632.000 -95.726 18.108 -124.546 -123.200 -117.754 -88.028 -80.247 -72.854   -67.109 -0.365     -1.556
Distance(mi)      7,728,394.000   0.562  1.777    0.000    0.000    0.000   0.030   0.464   6.955   441.750 20.386  1,649.595
Temperature(F)    7,564,541.000  61.663 19.014  -89.000   12.200   49.000  64.000  76.000  96.000   207.000 -0.514     -0.001
Wind_Chill(F)     5,729,375.000  58.251 22.390  -89.000   -2.000   43.000  62.000  75.000  96.000  

In [10]:
# %% 8. Categorical summary
rows = []
for c in cat_like:
    if c in ("ID", "Description", "Street"):   # high-cardinality / free text
        continue
    vc = df[c].value_counts(dropna=True)
    rows.append({
        "column": c,
        "unique": df[c].nunique(),
        "top": vc.index[0] if len(vc) else None,
        "top_freq": vc.iloc[0] if len(vc) else None,
        "top_%": vc.iloc[0] / len(df) * 100 if len(vc) else None,
    })
print("\nCATEGORICAL SUMMARY")
print(pd.DataFrame(rows).set_index("column"))



CATEGORICAL SUMMARY
                       unique          top  top_freq   top_%
column                                                      
Source                      3      Source1   4325632  55.971
Severity                    4            2   6156981  79.667
City                    13678        Miami    186917   2.419
County                   1871  Los Angeles    526851   6.817
State                      49           CA   1741433  22.533
Zipcode                825094        91761     11247   0.146
Country                     1           US   7728394 100.000
Timezone                    4   US/Eastern   3580167  46.325
Airport_Code             2045         KCQT    118332   1.531
Wind_Direction             24         CALM    961624  12.443
Weather_Condition         144         Fair   2560802  33.135
Sunrise_Sunset              2          Day   5334553  69.025
Civil_Twilight              2          Day   5695619  73.697
Nautical_Twilight           2          Day   6076156  78.621
Ast

In [11]:
# %% 9. Boolean (road feature) summary
bool_summary = pd.DataFrame({
    "True_count": df[bool_cols].sum(),
    "True_%": df[bool_cols].mean() * 100,
}).sort_values("True_%", ascending=False)
print("\nBOOLEAN FEATURES")
print(bool_summary)


BOOLEAN FEATURES
                 True_count  True_%
Traffic_Signal      1143772  14.800
Crossing             873763  11.306
Junction             571342   7.393
Stop                 214371   2.774
Station              201901   2.612
Amenity               96334   1.246
Railway               66979   0.867
Give_Way              36582   0.473
No_Exit               19545   0.253
Traffic_Calming        7598   0.098
Bump                   3514   0.045
Roundabout              249   0.003
Turning_Loop              0   0.000


In [12]:
# %% 10. Target variable: Severity
sev = df["Severity"].value_counts().sort_index()
print("\nSEVERITY DISTRIBUTION")
print(pd.DataFrame({"count": sev, "%": sev / sev.sum() * 100}))


SEVERITY DISTRIBUTION
            count      %
Severity                
1           67366  0.872
2         6156981 79.667
3         1299337 16.813
4          204710  2.649


In [13]:
# %% 11. Time and place coverage
print("\nACCIDENTS PER YEAR")
print(df["Start_Time"].dt.year.value_counts().sort_index())

print("\nTOP 10 STATES")
print(df["State"].value_counts().head(10))

print("\nTOP 10 CITIES")
print(df["City"].value_counts().head(10))

print("\nTOP 10 WEATHER CONDITIONS")
print(df["Weather_Condition"].value_counts().head(10))



ACCIDENTS PER YEAR
Start_Time
2016     410821
2017     718093
2018     893426
2019     954303
2020    1178913
2021    1563753
2022    1762452
2023     246633
Name: count, dtype: int64

TOP 10 STATES
State
CA    1741433
FL     880192
TX     582837
SC     382557
NY     347960
NC     338199
VA     303301
PA     296620
MN     192084
OR     179660
Name: count, dtype: int64

TOP 10 CITIES
City
Miami          186917
Houston        169609
Los Angeles    156491
Charlotte      138652
Dallas         130939
Orlando        109733
Austin          97359
Raleigh         86079
Nashville       72930
Baton Rouge     71588
Name: count, dtype: int64

TOP 10 WEATHER CONDITIONS
Weather_Condition
Fair                2560802
Mostly Cloudy       1016195
Cloudy               817082
Clear                808743
Partly Cloudy        698972
Overcast             382866
Light Rain           352957
Scattered Clouds     204829
Light Snow           128680
Fog                   99238
Name: count, dtype: int64


In [14]:
# %% 12. Data quality checks
duration_min = (df["End_Time"] - df["Start_Time"]).dt.total_seconds() / 60

print("\nDATA QUALITY CHECKS")
print(f"Negative durations (End < Start): {(duration_min < 0).sum():,}")
print(f"Zero-distance accidents         : {(df['Distance(mi)'] == 0).sum():,}")
print(f"Unparseable Start_Time          : {df['Start_Time'].isna().sum():,}")
print(f"Latitude outside US bounds      : "
      f"{((df['Start_Lat'] < 24) | (df['Start_Lat'] > 50)).sum():,}")
print(f"Longitude outside US bounds     : "
      f"{((df['Start_Lng'] < -125) | (df['Start_Lng'] > -66)).sum():,}")
print("\nAccident duration (minutes):")
print(duration_min.describe(percentiles=[.01, .5, .99]))



DATA QUALITY CHECKS
Negative durations (End < Start): 0
Zero-distance accidents         : 3,302,161
Unparseable Start_Time          : 0
Latitude outside US bounds      : 0
Longitude outside US bounds     : 0

Accident duration (minutes):
count   7,728,394.000
mean          444.421
std        13,512.293
min             1.217
1%             15.000
50%            74.833
99%           798.000
max     2,812,939.000
dtype: float64


In [15]:
# Outliers via the IQR rule
out_rows = []
for c in ["Temperature(F)", "Humidity(%)", "Pressure(in)", "Visibility(mi)",
          "Wind_Speed(mph)", "Precipitation(in)", "Distance(mi)"]:
    s = df[c].dropna()
    q1, q3 = s.quantile([.25, .75])
    iqr = q3 - q1
    n_out = ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum()
    out_rows.append({"column": c, "outliers": n_out, "outliers_%": n_out / len(s) * 100})
print("\nOUTLIERS (IQR rule)")
print(pd.DataFrame(out_rows).set_index("column"))



OUTLIERS (IQR rule)
                   outliers  outliers_%
column                                 
Temperature(F)        50515       0.668
Humidity(%)               0       0.000
Pressure(in)         443383       5.843
Visibility(mi)      1481065      19.613
Wind_Speed(mph)      210603       2.943
Precipitation(in)    533090       9.649
Distance(mi)         963606      12.468


In [16]:
# %% 13. Free-text column
desc_len = df["Description"].str.len()
print("\nDESCRIPTION LENGTH (characters)")
print(desc_len.describe())


DESCRIPTION LENGTH (characters)
count   7,728,389.000
mean           68.837
std            31.070
min             2.000
25%            47.000
50%            62.000
75%            85.000
max           679.000
Name: Description, dtype: float64


In [17]:
# %% 1. Imports and settings
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PATH = "US_Accidents_March23.csv"   # change to your file path
NROWS = None                        # e.g. 500_000 for a quick test
SAMPLE_N = 300_000                  # sample size for heavy plots
OUT_DIR = "figures"
os.makedirs(OUT_DIR, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
SEV_COLORS = {1: "#2ecc71", 2: "#f1c40f", 3: "#e67e22", 4: "#e74c3c"}


def save(fig, name):
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, f"{name}.png"), dpi=200, bbox_inches="tight")
    plt.show()

In [18]:
# %% 2. Load (skipped if `df` already exists from the description script)
try:
    df
    print("Using existing df")
except NameError:
    cat_cols = ["Source", "Severity", "State", "Country", "Timezone", "Airport_Code",
                "Wind_Direction", "Weather_Condition", "Sunrise_Sunset", "Civil_Twilight",
                "Nautical_Twilight", "Astronomical_Twilight"]
    df = pd.read_csv(PATH, dtype={c: "category" for c in cat_cols}, nrows=NROWS)
    for c in ["Start_Time", "End_Time", "Weather_Timestamp"]:
        df[c] = pd.to_datetime(df[c], format="mixed", errors="coerce")
    for c in df.select_dtypes("float64").columns:
        df[c] = pd.to_numeric(df[c], downcast="float")

df["Severity"] = df["Severity"].astype(int)
sample = df.sample(min(SAMPLE_N, len(df)), random_state=42)


Using existing df


In [ ]:
# %% 3. Missing values per column
miss = (df.isna().mean() * 100)
miss = miss[miss > 0].sort_values()

fig, ax = plt.subplots(figsize=(9, max(4, 0.35 * len(miss))))
ax.barh(miss.index, miss.values, color="#3498db")
for y, v in enumerate(miss.values):
    ax.text(v + 0.3, y, f"{v:.1f}%", va="center", fontsize=8)
ax.set(title="Missing Values by Column", xlabel="Missing (%)")
save(fig, "01_missing_values")


In [ ]:
# %% 4. Target variable: Severity distribution
sev = df["Severity"].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].bar(sev.index.astype(str), sev.values,
            color=[SEV_COLORS[i] for i in sev.index])
for x, v in zip(range(len(sev)), sev.values):
    axes[0].text(x, v, f"{v:,}\n({v / sev.sum() * 100:.1f}%)", ha="center", va="bottom", fontsize=8)
axes[0].set(title="Severity Distribution (Counts)", xlabel="Severity", ylabel="Accidents")
axes[0].set_ylim(0, sev.max() * 1.18)

axes[1].pie(sev.values, labels=[f"Severity {i}" for i in sev.index], autopct="%1.1f%%",
            colors=[SEV_COLORS[i] for i in sev.index], startangle=90,
            wedgeprops={"edgecolor": "white"})
axes[1].set_title("Severity Share")
save(fig, "02_severity_distribution")
